In [3]:
import pandas as pd

df = pd.read_csv('data/resqfood_listings.csv')
print(df.head())

  listing_id donor_id  donor_type    food_type  quantity   unit       city  \
0     L00001    D0655      Bakery     Beverage       6.6     kg    Chennai   
1     L00002    D0115      Bakery  Cooked Meal       4.7  boxes  Hyderabad   
2     L00003    D0026    HomeCook  Cooked Meal       0.5     kg  Hyderabad   
3     L00004    D0760      Bakery  Cooked Meal       4.2     kg     Mumbai   
4     L00005    D0282  Restaurant  Baked Goods       2.2  packs  Hyderabad   

      packaging          time_posted  time_since_cooked_mins  \
0      Airtight  2025-09-01 15:28:00                     184   
1    Cloth Wrap  2025-09-01 20:14:00                      34   
2  Covered Tray  2025-09-02 07:20:00                     136   
3    Sealed Box  2025-09-07 03:15:00                      65   
4  Covered Tray  2025-09-01 14:35:00                     124   

  donor_reliability  distance_km  ambient_temp_c  freshness_score  \
0               Low         8.07            26.7            0.276   
1       

In [5]:
df = df.drop_duplicates()

df['time_since_cooked_mins'] = pd.to_numeric(
    df['time_since_cooked_mins'], errors='coerce'
)

df = df.dropna()

In [8]:
total_requests = len(df)

completed = df[df['picked_up'] == 1].shape[0]
pending = df[df['picked_up'] == 0].shape[0]

# Average resolution time (in minutes)
avg_resolution = df[df['picked_up'] == 1]['time_since_cooked_mins'].mean()

# SLA: completed within 48 hours → convert to minutes
sla = df[
    (df['picked_up'] == 1) & 
    (df['time_since_cooked_mins'] <= 48 * 60)
].shape[0] / completed * 100 if completed != 0 else 0

print(total_requests, completed, pending, avg_resolution, sla)

5000 2839 2161 88.66607960549489 100.0


In [9]:
df['status'] = df['picked_up'].apply(
    lambda x: 'Completed' if x == 1 else 'Pending'
)

df['sla_status'] = df['time_since_cooked_mins'].apply(
    lambda x: 'Within SLA' if x <= 48 * 60 else 'Delayed'
)

In [11]:
df.to_csv("data/cleaned_data.csv", index=False)